In [ ]:
CREATE DATABASE ELECTRICITY_PRICES;
CREATE SCHEMA PRICES;

CREATE OR REPLACE TABLE prices (
    timestamp TIMESTAMP,
    fixing_i_price FLOAT,
    fixing_i_volume FLOAT,
    fixing_ii_price FLOAT,
    fixing_ii_volume FLOAT
);

CREATE STAGE my_stage;


//dodanie plików do stage

COPY INTO prices
FROM @my_stage/electricity_prices.csv
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1 
               FIELD_DELIMITER = ',' TIMESTAMP_FORMAT = 'DD.MM.YYYY HH24:MI');

select * from prices;               

SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(timestamp IS NULL) AS null_timestamp,
    COUNT_IF(fixing_i_price IS NULL) AS null_fixing_i_price,
    COUNT_IF(fixing_i_volume IS NULL) AS null_fixing_i_volume,
    COUNT_IF(fixing_ii_price IS NULL) AS null_fixing_ii_price,
    COUNT_IF(fixing_ii_volume IS NULL) AS null_fixing_ii_volume
FROM prices;

UPDATE prices
SET fixing_i_price = fixing_ii_price
WHERE fixing_i_price IS NULL;

SELECT
    DATE_TRUNC('DAY', timestamp) AS day,
    AVG(fixing_i_price) AS avg_fixing_i_price
FROM prices
GROUP BY 1
ORDER BY 1;



SELECT *
FROM prices
ORDER BY fixing_i_price DESC
LIMIT 10;

ALTER ACCOUNT SET ENABLE_CUSTOM_SQL_FOR_DASHBOARDS = TRUE;
ALTER ACCOUNT SET ENABLE_DASHBOARD_CONTROLS = TRUE;

SHOW PARAMETERS IN ACCOUNT;

SELECT SYSTEM$SHOW_PREVIEW_FEATURES();

SHOW PARAMETERS IN ACCOUNT;


CREATE OR REPLACE TABLE profile_standard_raw (
    dzien_tygodnia STRING,
    data STRING,
    h1 STRING, h2 STRING, h3 STRING, h4 STRING, h5 STRING, h6 STRING,
    h7 STRING, h8 STRING, h9 STRING, h10 STRING, h11 STRING, h12 STRING,
    h13 STRING, h14 STRING, h15 STRING, h16 STRING, h17 STRING, h18 STRING,
    h19 STRING, h20 STRING, h21 STRING, h22 STRING, h23 STRING, h24 STRING,
    suma_dnia STRING
);

COPY INTO profile_standard_raw
FROM @my_stage/plik_2021_2024_profil_standardowy2.csv
FILE_FORMAT = (TYPE='CSV' FIELD_DELIMITER=';' SKIP_HEADER=1 FIELD_OPTIONALLY_ENCLOSED_BY='"')
ON_ERROR='CONTINUE';

select * from profile_standard_raw;

CREATE OR REPLACE TABLE profile_standard AS
SELECT
    dzien_tygodnia,
    TO_DATE(data, 'DD.MM.YYYY') AS data,
    TRY_TO_DOUBLE(REPLACE(h1, ',', '.')) AS h1,
    TRY_TO_DOUBLE(REPLACE(h2, ',', '.')) AS h2,
    TRY_TO_DOUBLE(REPLACE(h3, ',', '.')) AS h3,
    TRY_TO_DOUBLE(REPLACE(h4, ',', '.')) AS h4,
    TRY_TO_DOUBLE(REPLACE(h5, ',', '.')) AS h5,
    TRY_TO_DOUBLE(REPLACE(h6, ',', '.')) AS h6,
    TRY_TO_DOUBLE(REPLACE(h7, ',', '.')) AS h7,
    TRY_TO_DOUBLE(REPLACE(h8, ',', '.')) AS h8,
    TRY_TO_DOUBLE(REPLACE(h9, ',', '.')) AS h9,
    TRY_TO_DOUBLE(REPLACE(h10, ',', '.')) AS h10,
    TRY_TO_DOUBLE(REPLACE(h11, ',', '.')) AS h11,
    TRY_TO_DOUBLE(REPLACE(h12, ',', '.')) AS h12,
    TRY_TO_DOUBLE(REPLACE(h13, ',', '.')) AS h13,
    TRY_TO_DOUBLE(REPLACE(h14, ',', '.')) AS h14,
    TRY_TO_DOUBLE(REPLACE(h15, ',', '.')) AS h15,
    TRY_TO_DOUBLE(REPLACE(h16, ',', '.')) AS h16,
    TRY_TO_DOUBLE(REPLACE(h17, ',', '.')) AS h17,
    TRY_TO_DOUBLE(REPLACE(h18, ',', '.')) AS h18,
    TRY_TO_DOUBLE(REPLACE(h19, ',', '.')) AS h19,
    TRY_TO_DOUBLE(REPLACE(h20, ',', '.')) AS h20,
    TRY_TO_DOUBLE(REPLACE(h21, ',', '.')) AS h21,
    TRY_TO_DOUBLE(REPLACE(h22, ',', '.')) AS h22,
    TRY_TO_DOUBLE(REPLACE(h23, ',', '.')) AS h23,
    TRY_TO_DOUBLE(REPLACE(h24, ',', '.')) AS h24,
    TRY_TO_DOUBLE(REPLACE(suma_dnia, ',', '.')) AS suma_dnia
FROM profile_standard_raw;

select * from profile_standard;



CREATE OR REPLACE TABLE profile_standard_unpivot AS
SELECT
  data, 
  TO_NUMBER(REPLACE(UPPER(hour_col), 'H', '')) AS hour_num,
  TRY_TO_DOUBLE(REPLACE(TO_VARCHAR(value), ',', '.')) AS profile_value
FROM profile_standard
UNPIVOT (
  value FOR hour_col IN (
    h1, h2, h3, h4, h5, h6,
    h7, h8, h9, h10, h11, h12,
    h13, h14, h15, h16, h17, h18,
    h19, h20, h21, h22, h23, h24
  )
);


CREATE OR REPLACE TABLE prices_2 AS
SELECT
  p.timestamp,
  p.fixing_i_price,
  p.fixing_i_volume,
  p.fixing_ii_price,
  p.fixing_ii_volume,
  ps.profile_value
FROM prices p
JOIN profile_standard_unpivot ps
  ON TO_DATE(p.timestamp) = ps.data
  AND (EXTRACT(HOUR FROM p.timestamp) + 1) = ps.hour_num
WHERE p.timestamp >= '2021-01-01' 
  AND p.timestamp <  '2025-01-01';


select * from prices_2;

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import streamlit as st
from datetime import timedelta
import math
import altair as alt
import calendar

session = get_active_session()

In [ ]:
df = session.sql("""
    SELECT 
        TIMESTAMP,
        FIXING_I_PRICE,
        PROFILE_VALUE
    FROM prices_2
""").to_pandas()

df['DATE'] = pd.to_datetime(df['TIMESTAMP'])

def classify_prices(df):
    prices = df['FIXING_I_PRICE'].to_numpy()
    n = len(df)

    labels = np.empty(n, dtype=object)
    window_past = 24
    window_future = 24

    for i in range(n):
        start_idx = max(0, i - window_past)
        end_idx = min(n, i + window_future + 1)
        local_prices = prices[start_idx:end_idx]

        min_price = local_prices.min()
        max_price = local_prices.max()
        q_10 = np.quantile(local_prices, 0.10)
        q_25 = np.quantile(local_prices, 0.25)
        q_75 = np.quantile(local_prices, 0.75)

        current_price = prices[i]
        
        if current_price <= q_10:
            labels[i] = "najtaniej"
        elif current_price <= q_25 and current_price > q_10:
            labels[i] = "tanio"
        elif current_price >= q_75:
            labels[i] = "drogo"
        else:
            labels[i] = "średnio"

    df['pom'] = labels
    return df


def simulate_costs_with_battery(df):
    df = df.copy()
    df['DAY'] = df['DATE'].dt.date
    df['HOUR'] = df['DATE'].dt.hour

    # roczne zużycie // kWh 
    annual_usage_kwh = monthly_usage * 12

    # normalizujemy profil standardowy w ramach roku
    df['YEAR'] = df['DATE'].dt.year
    df['profile_norm'] = df.groupby('YEAR')['PROFILE_VALUE'].transform(lambda x: x / x.sum())

    # liczba godzin w roku (przestepny?)
    df['hours_in_year'] = df['YEAR'].apply(lambda y: 366*24 if calendar.isleap(y) else 365*24)

    # godzinowa konsumpcja wg profilu
    df['hourly_usage_kwh'] = df['profile_norm'] * annual_usage_kwh

    standby_consumption_kwh_per_hour = standby_consumption_kwh_per_day / 24

    results = []
    battery_status = 0.0                 # kWh zmagazynowane
    battery_energy_price = 0.0              # PLN/kWh (średni koszt zmagazynowanej energii)

    for index, row in df.iterrows():
        markt_price = row['FIXING_I_PRICE']  # PLN/MWh
        market_price = markt_price / 1000   # PLN/kWh

        charge_amount = 0.0      # energia kupiona /kWh
        discharge_amount = 0.0   # energia użyta /kWh
        charge_cost = 0.0        # PLN zapłacone za ładowanie
        saving = 0.0             # PLN zaoszczędzone za użycie z magazynu

        standby_cost = standby_consumption_kwh_per_hour * market_price   # stały koszt magazynu

        # ładowanie: gdy najtaniej, lub gdy tanio i mamy mniej niż 80%
        if battery_status < capacity_kwh and row['pom'] in ["najtaniej", "tanio"]:
            if row['pom'] == "najtaniej": 
                max_capacity = capacity_kwh 
                can_charge = min(charge_power_kw, max_capacity - battery_status)
            else:
                max_capacity = 0.8 * capacity_kwh
                can_charge = min(charge_power_kw, max_capacity - battery_status)
            if can_charge > 0:
                charge_amount = can_charge
                charge_cost = market_price * charge_amount  # PLN
                battery_energy_price = (battery_status * battery_energy_price + charge_cost / (battery_status + charge_amount)) 
                battery_status += charge_amount 

        # rozładowanie: gdy drogo, lub średnio jeśli się opłaca (koszt w baterii < rynkowa cena)
        elif battery_status > 0 and (
            row['pom'] == "drogo"
            or (row['pom'] == "średnio" and battery_energy_price < market_price)
        ):
            discharge_amount = min(row['hourly_usage_kwh'], battery_status)
            if discharge_amount > 0:
                battery_status -= discharge_amount
                # oszczędność = niekupiona energia * cena
                saving = market_price * discharge_amount

        # netto całego 'kroku'
        net_saving = saving - charge_cost - standby_cost

        results.append({
            'PERIOD': row['DATE'],
            'saving': saving,
            'charge_cost': charge_cost,
            'standby_cost': standby_cost,
            'net_saving': net_saving,
            'battery_status': battery_status
        })

    result_df = pd.DataFrame(results)
    agg_df = result_df.copy()

    if time_frame == 'Daily':
        agg_df['PERIOD'] = agg_df['PERIOD'].dt.date
    elif time_frame == 'Weekly':
        agg_df['PERIOD'] = agg_df['PERIOD'].dt.to_period('W').dt.start_time
    elif time_frame == 'Monthly':
        agg_df['PERIOD'] = agg_df['PERIOD'].dt.to_period('M').dt.to_timestamp()
    elif time_frame == 'Quarterly':
        agg_df['PERIOD'] = agg_df['PERIOD'].dt.to_period('Q').dt.to_timestamp()

    agg_df = agg_df.groupby('PERIOD').agg(
        net_saving=('net_saving', 'sum'),
        total_saving=('saving', 'sum'),
        total_charge_cost=('charge_cost', 'sum'),
        total_standby_cost=('standby_cost', 'sum'),
        last_battery_status=('battery_status', 'last')
    ).reset_index()

    result_df['hourly_usage_kwh'] = df['hourly_usage_kwh'].values

    return agg_df, result_df


def create_chart(df, x, y, color, chart_type, height=250):
    if chart_type == "Bar":
        st.bar_chart(df, x=x, y=y, color=color, height=height)
    else:
        st.area_chart(df, x=x, y=y, color=color, height=height)

df = classify_prices(df) 
df.head(30)

In [ ]:
col = st.columns(4)

with col[0]:
    start_date = st.date_input("Start date", df['DATE'].min().date())
with col[1]:
    end_date = st.date_input("End date", df['DATE'].max().date())
with col[2]:
    time_frame = st.selectbox("Select time frame", ("Daily", "Weekly", "Monthly", "Quarterly"), index=2)
with col[3]:
    chart_type = st.selectbox("Select chart type", ("Bar", "Area"), index=0)

monthly_usage = st.slider("Miesięczne zużycie prądu (kWh)", min_value=50, max_value=1000, value=250, step=10)
monthly_usage_kwh = monthly_usage / 1000

mask = (df['DATE'].dt.date >= start_date) & (df['DATE'].dt.date <= end_date)
df_filtered = df.loc[mask].copy()

if time_frame == 'Daily':
    df_filtered['PERIOD'] = df_filtered['DATE'].dt.date
elif time_frame == 'Weekly':
    df_filtered['PERIOD'] = df_filtered['DATE'].dt.to_period('W').dt.start_time
elif time_frame == 'Monthly':
    df_filtered['PERIOD'] = df_filtered['DATE'].dt.to_period('M').dt.to_timestamp()
elif time_frame == 'Quarterly':
    df_filtered['PERIOD'] = df_filtered['DATE'].dt.to_period('Q').dt.to_timestamp()

df_filtered['DATE_ONLY'] = df_filtered['DATE'].dt.date
period_days = df_filtered.groupby('PERIOD')['DATE_ONLY'].nunique().reset_index()
period_days = period_days.rename(columns={'DATE_ONLY': 'days_in_period'})

df_agg = df_filtered.groupby('PERIOD').agg(avg_price=('FIXING_I_PRICE', 'mean')).reset_index()
df_agg = df_agg.merge(period_days, on='PERIOD', how='left')

df_agg['estimated_cost'] = df_agg['avg_price'] * (df_agg['days_in_period'] / 30) * monthly_usage_kwh

days_x = (end_date - start_date).days
months_x = days_x * 12 / 365.25
avg_price = df_filtered['FIXING_I_PRICE'].mean()
total_cost = avg_price * months_x * monthly_usage_kwh

cols = st.columns(3)
with cols[0]:
    st.metric("Średnia cena [zł/MWh]", f"{df_agg['avg_price'].mean():.2f}")
with cols[1]:
    st.metric("Łączny koszt [zł]", f"{total_cost:.2f}")
with cols[2]:
    st.metric("Okres (dni)", (end_date - start_date).days)

st.divider()

st.subheader(f"Średni koszt przy zużyciu {monthly_usage} kWh")
create_chart(df=df_agg, x="PERIOD", y="estimated_cost", color="#48CAE4", chart_type=chart_type)

st.subheader("Parametry magazynu energii")
with st.expander("Wprowadź parametry magazynu"):
    col = st.columns(4)
    with col[0]:
        capacity_kwh = st.slider("Pojemność (kWh)", 1.0, 20.0, 5.0, step=0.5)
    with col[1]:
        charge_power_kw = st.slider("Maks. moc ładowania (kW)", 1.0, 11.0, 3.0, step=0.5)
    with col[2]:
        efficiency = st.number_input("Efektywność (%)", value=90)
    with col[3]:
        standby_power_w = st.number_input("Stały pobór mocy (W)", value=15)

standby_consumption_kwh_per_day = standby_power_w * 24 / 1000

df_agg['cost_without_battery'] = df_agg['avg_price'] * (df_agg['days_in_period'] / 30) * monthly_usage_kwh


#symulacja
battery_costs, df_battery_detail = simulate_costs_with_battery(df_filtered)

#agregacja czasowa
df_agg = df_agg.merge(battery_costs[['PERIOD', 'net_saving']], on='PERIOD', how='left')

df_agg['net_saving'] = df_agg['net_saving'].fillna(0)

df_agg['cost_with_battery'] = df_agg['estimated_cost'] - df_agg['net_saving']

#wykres2
st.subheader(f"Wydatki przy użyciu magazynu ({capacity_kwh} kWh)")
create_chart(df=df_agg, x="PERIOD", y="cost_with_battery", color="#D62728", chart_type=chart_type)

#wykres3
st.subheader("Oszczędności dzięki magazynowi energii")
create_chart(df=df_agg, x="PERIOD", y="net_saving", color="#33B864", chart_type=chart_type)

total_savings = df_agg['net_saving'].sum()
st.markdown(f"**Łączne oszczędności w okresie:** {total_savings:.2f} zł")


st.subheader("Proporcje oszczędności dzięki magazynowi energii")
bar_cost = alt.Chart(df_agg).mark_bar(color='#D62728').encode(
    x=alt.X('PERIOD:T', title='Okres'),
    y=alt.Y('cost_with_battery:Q', title='Koszt (zł)')
)

bar_saving = alt.Chart(df_agg).mark_bar(color='#33B864').encode(
    x=alt.X('PERIOD:T'),
    y=alt.Y('cost_with_battery:Q'),
    y2=alt.Y2('estimated_cost:Q')
)

final_chart = (bar_cost + bar_saving).properties(height=250)
st.altair_chart(final_chart, use_container_width=True)

# top 5 dni o największym zużyciu
daily_usage = df_battery_detail.groupby(df_battery_detail['PERIOD'].dt.date)['hourly_usage_kwh'].sum().reset_index()
daily_usage.columns = ['DAY', 'daily_usage_kwh']
top5_days = daily_usage.nlargest(5, 'daily_usage_kwh')

st.subheader("Top 5 dni o największym zużyciu (wg profilu standardowego)")
st.dataframe(top5_days)